In [0]:
%pip install -U "datasets==2.18.0" "huggingface_hub==0.23.5" pandas numpy sentence-transformers faiss-cpu langchain langchain-text-splitters
dbutils.library.restartPython()

In [0]:
import pandas as pd 
import numpy as np 
import faiss
import pickle
import re
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [0]:
dataset = load_dataset("theatticusproject/cuad")
dataset

In [0]:
df = dataset["train"].to_pandas()

df.head()

In [0]:
df.shape

In [0]:
df.columns

In [0]:
df["text_length"] = df["text"].astype(str).apply(len)
df[["text", "text_length"]].head()

In [0]:
df["text_length"].describe()

In [0]:

def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text)           
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

df[["clean_text", "text_length"]].head()

In [0]:
df = df[df["clean_text"].str.len() > 20]
print("Rows remaining:", len(df))

In [0]:
for i in range(10):
    print(f"\n----- ROW {i} -----")
    print(df.iloc[i]["clean_text"][:1000])

In [0]:
df["clean_text"].str.len().describe()

In [0]:
print(df.iloc[100]["clean_text"])

In [0]:
print(df.iloc[1000]["clean_text"])

In [0]:
print(df.iloc[5000]["clean_text"])

In [0]:
print(df.iloc[10000]["clean_text"])

In [0]:
df.sort_values("text_length", ascending=False)[["clean_text","text_length"]].head(10)

In [0]:
df = df[
    (df["clean_text"].str.len() > 50)
]

print("Rows Remaining:", len(df))

In [0]:
metadata_patterns = [
    "CONTRACT UNDERSTANDING ATTICUS DATASET",
    "CUAD",
    "README",
    "Answer Format:",
]

for pattern in metadata_patterns:
    df = df[~df["clean_text"].str.contains(pattern, case=False, na=False)]

print("Rows remaining after metadata removal:", len(df))

In [0]:
df_clean = df.copy()
df_clean.shape

In [0]:
df_clean = df_clean[
    ~df_clean["clean_text"].str.startswith("Source:", na=False)
]

print("Rows after source removal:", len(df_clean))

In [0]:
df_clean.sample(10)[["clean_text","text_length"]]

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=150
)

chunks = []

for text in df_clean["clean_text"]:
    split_chunks = text_splitter.split_text(text)
    chunks.extend(split_chunks)

print("Total chunks:", len(chunks))

In [0]:
print(chunks[0])
print("\n\n")
print(chunks[1])
print("\n\n")
print(chunks[2])

In [0]:
df_clean = df_clean[
    ~df_clean["clean_text"].str.contains("master clauses CSV|SQuAD-style JSON|28 Excels|Category List", case=False, na=False)
]

print("Rows after documentation cleanup:", len(df_clean))

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=150
)

chunks = []

for text in df_clean["clean_text"]:
    split_chunks = text_splitter.split_text(text)
    chunks.extend(split_chunks)

print("Total chunks:", len(chunks))

In [0]:
for _ in range(5):
    idx = random.randint(0, len(chunks)-1)
    print(chunks[idx][:1000])
    print("\n")

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [0]:
embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

In [0]:
embeddings.shape

In [0]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Vectors indexed:", index.ntotal)

In [0]:
faiss.write_index(
    index,
    "cuad_faiss_index.index"
)

print("FAISS index saved.")

In [0]:
with open("cuad_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Chunks saved.")

In [0]:
np.save(
    "cuad_embeddings.npy",
    embeddings
)

print("Embeddings saved.")

In [0]:
query = "termination clause"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for idx in indices[0]:
    print("\n")
    print(chunks[idx][:1000])

In [0]:
query = "confidentiality agreement"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for idx in indices[0]:
    print("\n")
    print(chunks[idx][:1000])

In [0]:
query = "indemnification"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for idx in indices[0]:
    print("\n")
    print(chunks[idx][:1000])

In [0]:
query = "termination clause"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
    print(f"\nResult {rank}")
    print("Distance: {distance}")
    print(chunks[idx][:500])

In [0]:
def search_contracts(query, k=5):
    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=k
    )

    results = []

    for idx, distance in zip(indices[0], distances[0]):
        results.append({
            "distance": float(distance),
            "text": chunks[idx]
        })

    return results

In [0]:
results = search_contracts("termination clause")

for r in results:
    print("\nDistance:", r["distance"])
    print(r["text"][:500])

In [0]:
print("CUAD Pipeline Summary\n")
print("Contracts after cleaning:", len(df_clean))
print("Total chunks:", len(chunks))
print("Embedding dimensions:", embeddings.shape[1])
print("Vectors indexed:", index.ntotal)